In [212]:
import os, sys, time
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
pd.options.mode.chained_assignment = None
import h5py
import sqlalchemy
from shapely import wkt
import geopandas as gpd
import seaborn as sns
from itertools import cycle, islice
import pyodbc
import warnings
import matplotlib.pyplot as plt
import psrcelmerpy
import numpy as np
from scipy.spatial import cKDTree

In [213]:
df_lu = pd.read_csv(r'C:\Workspace\displacement_index\parcels_urbansim.txt',
                   delim_whitespace=True)

# Load as a geodataframe
gdf_lu = gpd.GeoDataFrame(
    df_lu, geometry=gpd.points_from_xy(df_lu.xcoord_p, df_lu.ycoord_p))

crs = {'init' : 'EPSG:2285'}
gdf_lu.crs = crs
base_year = "2023"

parcel_geog = pd.read_sql_table('parcel_'+base_year+'_geography', 'sqlite:///N:/rtp_2026_2050/final_runs/sc_base_year_2023_final/soundcast/inputs/db/soundcast_inputs_2023.db')

In [214]:
parcel_geog = parcel_geog.drop("geometry", axis=1)

In [ ]:
new_df_lu = gdf_lu.merge(parcel_geog,left_on='parcelid', right_on='ParcelID', how='left')

In [216]:
eg_conn = psrcelmerpy.ElmerGeoConn()
schools_gdf = eg_conn.read_geolayer('public_schools')

In [217]:
crs = {'init' : 'EPSG:2285'}
schools_gdf = schools_gdf.to_crs(crs)

In [218]:
school_district_gdf = eg_conn.read_geolayer('school_districts')
school_district_gdf = school_district_gdf[["lea_code", "short_name", "lea_name", "Shape", "geometry"]]

In [219]:
crs = {'init' : 'EPSG:2285'}
school_district_gdf = school_district_gdf.to_crs(crs)

In [220]:
h5_file = h5py.File(r'N:\rtp_2026_2050\final_runs\sc_base_year_2023_final\soundcast\inputs\scenario\landuse\hh_and_persons.h5', 'r')
hh_df = pd.DataFrame()
h5_file['Household']['hhtaz'][0]
for col in h5_file['Household'].keys():
    hh_df[col] = h5_file['Household'][col][:]
h5_file.close()

In [221]:
df = hh_df[['hhparcel','hhsize']].groupby('hhparcel').sum()
df.rename(columns={'hhsize':'population'}, inplace=True)
print(df)

          population
hhparcel            
21                 7
27                92
70                88
71                 6
83                 2
...              ...
1327519            3
1327520            2
1327524            8
1327540            7
1327542            2

[1054131 rows x 1 columns]


In [222]:
new_df_lu.columns

Index(['aparks', 'empedu_p', 'empfoo_p', 'empgov_p', 'empind_p', 'empmed_p',
       'empofc_p', 'empoth_p', 'empret_p', 'emprsc_p', 'empsvc_p', 'emptot_p',
       'hh_p', 'lutype_p', 'mfunits', 'nparks', 'parcelid', 'parkdy_p',
       'parkhr_p', 'ppricdyp', 'pprichrp', 'sfunits', 'sqft_p', 'stugrd_p',
       'stuhgh_p', 'stuuni_p', 'taz_p', 'xcoord_p', 'ycoord_p', 'geometry',
       'Unnamed: 0', 'ParcelID', 'GEOID20', 'Census2020BlockGroup',
       'Census2020Tract', 'Census2020Block', 'rg_proposed', 'CityName',
       'CountyName', 'TAZ', 'District', 'district_name', 'GrowthCenterName',
       'mic', 'place_name_2020', 'L0ElmerGeo_DBO_tract2020_nowater_geoid20',
       'equity_focus_areas_2023__efa_poc',
       'equity_focus_areas_2023__efa_pov200',
       'equity_focus_areas_2023__efa_lep',
       'equity_focus_areas_2023__efa_youth',
       'equity_focus_areas_2023__efa_older',
       'equity_focus_areas_2023__efa_dis', 'BaseYear', 'all_day_transit',
       'frequent_transit', 'hc

In [80]:
df.columns

Index(['population'], dtype='object')

In [81]:
len(new_df_lu)

1329928

In [223]:
new_df_lu = new_df_lu.merge(df, left_on='parcelid', right_on='hhparcel', how='inner')

In [224]:
len(new_df_lu)

1054131

In [225]:
new_df_lu.columns

Index(['aparks', 'empedu_p', 'empfoo_p', 'empgov_p', 'empind_p', 'empmed_p',
       'empofc_p', 'empoth_p', 'empret_p', 'emprsc_p', 'empsvc_p', 'emptot_p',
       'hh_p', 'lutype_p', 'mfunits', 'nparks', 'parcelid', 'parkdy_p',
       'parkhr_p', 'ppricdyp', 'pprichrp', 'sfunits', 'sqft_p', 'stugrd_p',
       'stuhgh_p', 'stuuni_p', 'taz_p', 'xcoord_p', 'ycoord_p', 'geometry',
       'Unnamed: 0', 'ParcelID', 'GEOID20', 'Census2020BlockGroup',
       'Census2020Tract', 'Census2020Block', 'rg_proposed', 'CityName',
       'CountyName', 'TAZ', 'District', 'district_name', 'GrowthCenterName',
       'mic', 'place_name_2020', 'L0ElmerGeo_DBO_tract2020_nowater_geoid20',
       'equity_focus_areas_2023__efa_poc',
       'equity_focus_areas_2023__efa_pov200',
       'equity_focus_areas_2023__efa_lep',
       'equity_focus_areas_2023__efa_youth',
       'equity_focus_areas_2023__efa_older',
       'equity_focus_areas_2023__efa_dis', 'BaseYear', 'all_day_transit',
       'frequent_transit', 'hc

In [226]:
len(new_df_lu[["Census2020Tract", "parcelid", "geometry", "population"]])

1054131

In [227]:
school_district_gdf = school_district_gdf.copy()
new_df_lu =  new_df_lu.copy()
print(new_df_lu.total_bounds)
print(school_district_gdf.total_bounds)

print(len(new_df_lu))
print(len(school_district_gdf))


parcel_school_district = gpd.sjoin(
    new_df_lu[["Census2020Tract", "parcelid", "geometry", "population"]],  # keep only what you need
    school_district_gdf[['lea_name', 'geometry']],  # keep only what you need
    how='left',     
    predicate='intersects'   # or 'intersects'
)
parcel_school_district

[1100514.10378    -93023.8403113 1559795.69283    476702.458662 ]
[1095631.85983406  -97406.58393161 1622341.8920418   506798.53114656]
1054131
53


,Census2020Tract,parcelid,geometry,population,index_right,lea_name
0,5.303303e+10,21,POINT (1296020.776 133570.225),7,17.0,Kent School District
1,5.303303e+10,27,POINT (1294922.462 122771.109),92,11.0,Auburn School District
2,5.303303e+10,70,POINT (1297479.914 121760.485),88,11.0,Auburn School District
3,5.303303e+10,71,POINT (1297784.084 122054.965),6,11.0,Auburn School District
4,5.303303e+10,83,POINT (1296948.986 121971.059),2,11.0,Auburn School District
...,...,...,...,...,...,...
1054126,5.306105e+10,1327519,POINT (1468946.03 466268.838),3,50.0,Darrington School District
1054127,5.306105e+10,1327520,POINT (1467305.952 466225.806),2,50.0,Darrington School District
1054128,5.306105e+10,1327524,POINT (1470382.329 469427.38),8,50.0,Darrington School District
1054129,5.306105e+10,1327540,POINT (1464092.68 463997.791),7,50.0,Darrington School District


In [228]:
parcel_dict = {
    name: group
    for name, group in parcel_school_district.groupby("lea_name")
}

In [ ]:
parcel_dict["Arlington School District"]

,Census2020Tract,parcelid,geometry,population,index_right,lea_name
843554,5.306105e+10,1062044,POINT (1325575.152 438403.362),2,43.0,Arlington School District
843555,5.306105e+10,1062045,POINT (1325573.807 438308.313),1,43.0,Arlington School District
843556,5.306105e+10,1062046,POINT (1325571.975 438251.368),5,43.0,Arlington School District
843557,5.306105e+10,1062047,POINT (1325570.329 438194.491),2,43.0,Arlington School District
843558,5.306105e+10,1062048,POINT (1325568.057 438118.53),4,43.0,Arlington School District
...,...,...,...,...,...,...
1053512,5.306105e+10,1325907,POINT (1368364.865 449437.94),2,43.0,Arlington School District
1053513,5.306105e+10,1325909,POINT (1369359.978 451248.461),4,43.0,Arlington School District
1053514,5.306105e+10,1325914,POINT (1367364.243 448382.979),2,43.0,Arlington School District
1053515,5.306105e+10,1325915,POINT (1367679.443 448379.128),2,43.0,Arlington School District


In [230]:
school_district_gdf['lea_code'] = school_district_gdf['lea_code'].astype(str)
schools_gdf['lea_code'] = schools_gdf['lea_code'].astype(str)
school_merge = school_district_gdf.merge(schools_gdf, left_on='lea_code', right_on='lea_code', how='left')

In [231]:
school_merge.columns

Index(['lea_code', 'short_name', 'lea_name_x', 'Shape_x', 'geometry_x',
       'OBJECTID', 'object_id', 'school_code', 'school', 'single_address',
       'esd_code', 'esd_name', 'lea_name_y', 'school_category', 'ayp_code',
       'grade_category', 'principal', 'phone', 'email', 'lowest_grade',
       'highest_grade', 'mailing_address', 'congression', 'legislative',
       'county', 'geocoded_y', 'geocoded_x', 'nces_y', 'nces_x', 'adjusted_l',
       'SDE_STATE_ID', 'Shape_y', 'geometry_y'],
      dtype='object')

In [232]:
school_merge = school_merge.rename(columns={
    "geometry_x": "district_geometry", 
    "geometry_y": "school_geometry"})

In [233]:
school_merge

,lea_code,short_name,lea_name_x,Shape_x,district_geometry,OBJECTID,object_id,school_code,school,single_address,...,legislative,county,geocoded_y,geocoded_x,nces_y,nces_x,adjusted_l,SDE_STATE_ID,Shape_y,school_geometry
0,17001,Seattle,Seattle Public Schools,POLYGON ((1247383.7620383054 271911.6397008150...,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",215,466,2450,Daniel Bagley Elementary School,"7821 Stone Ave N, Seattle, WA 98103",...,Legislative District 46,King,47.68646499,-122.34235598,47.68637900,-122.34239800,46,0,POINT (1268752.0257321447 254011.92714072764),POINT (1268752.026 254011.927)
1,17001,Seattle,Seattle Public Schools,POLYGON ((1247383.7620383054 271911.6397008150...,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",192,406,2199,Concord International School,"723 S Concord St, Seattle, WA 98108",...,Legislative District 34,King,47.52352498,-122.32450403,47.52360000,-122.32450000,34,0,POINT (1272009.1376562268 194561.0230768919),POINT (1272009.138 194561.023)
2,17001,Seattle,Seattle Public Schools,POLYGON ((1247383.7620383054 271911.6397008150...,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",1022,2256,3974,Thornton Creek Elementary School,"7712 40th Ave NE, Seattle, WA 98115",...,Legislative District 46,King,47.68573300,-122.28384996,47.68528700,-122.28465100,46,0,POINT (1282965.3303107172 253340.11549222469),POINT (1282965.33 253340.115)
3,17001,Seattle,Seattle Public Schools,POLYGON ((1247383.7620383054 271911.6397008150...,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",859,1925,2285,Roosevelt High School,"1410 NE 66th St, Seattle, WA 98115",...,Legislative District 46,King,47.67746400,-122.31303700,47.67730000,-122.31280000,46,0,POINT (1275977.1618723869 250559.28443972766),POINT (1275977.162 250559.284)
4,17001,Seattle,Seattle Public Schools,POLYGON ((1247383.7620383054 271911.6397008150...,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",713,1539,3218,North Beach Elementary School,"9018 24th Ave NW, Seattle, WA 98117",...,Legislative District 36,King,47.69494299,-122.38710604,47.69480000,-122.38710000,36,0,POINT (1257805.8373227268 257302.08558663726),POINT (1257805.837 257302.086)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1078,31401,Stanwood-Camano,Stanwood-Camano School District,POLYGON ((1246446.8410118818 477516.3758002221...,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",1041,2312,4364,Twin City Elementary School,"26211 72nd Ave NW, Stanwood, WA 98292",...,Legislative District 10,Snohomish,48.23470101,-122.32778597,48.23450000,-122.32880000,10,0,POINT (1275985.7891517282 453850.21873630583),POINT (1275985.789 453850.219)
1079,31401,Stanwood-Camano,Stanwood-Camano School District,POLYGON ((1246446.8410118818 477516.3758002221...,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",955,2142,3125,Stanwood Elementary School,"10227 273rd Pl NW, Stanwood, WA 98292",...,Legislative District 10,Snohomish,48.24504801,-122.37268397,48.24510000,-122.37300000,10,0,POINT (1265293.3410484642 457928.64463455975),POINT (1265293.341 457928.645)
1080,31401,Stanwood-Camano,Stanwood-Camano School District,POLYGON ((1246446.8410118818 477516.3758002221...,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",801,1782,4512,Port Susan Middle School,"7506 267th St NW, Stanwood, WA 98292",...,Legislative District 10,Snohomish,48.23717300,-122.33566898,48.23720000,-122.33570000,10,0,POINT (1274323.748840645 454867.71932597458),POINT (1274323.749 454867.719)
1081,31401,Stanwood-Camano,Stanwood-Camano School District,POLYGON ((1246446.8410118818 477516.3758002221...,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",140,279,4513,Cedarhome Elementary School,"27911 68th Ave NW, Stanwood, WA 98292",...,Legislative District 10,Snohomish,48.24947101,-122.32227604,48.24940000,-122.32360000,10,0,POINT (1277358.0829950571 459260.01992456615),POINT (1277358.083 459260.02)


In [94]:
school_merge["school_geometry"].isna().sum()

np.int64(0)

In [234]:
school_dict = {
    district: group
    for district, group in school_merge.groupby("lea_name_x")
}

In [235]:
school_dict["Arlington School District"]

,lea_code,short_name,lea_name_x,Shape_x,district_geometry,OBJECTID,object_id,school_code,school,single_address,...,legislative,county,geocoded_y,geocoded_x,nces_y,nces_x,adjusted_l,SDE_STATE_ID,Shape_y,school_geometry
987,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",793,1758,4573,Pioneer Elementary School (Arlington),"8213 Eaglefield Dr, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.16572101,-122.12065296,48.16560000,-122.12060000,10,0,POINT (1326297.1193031371 427802.45127815008),POINT (1326297.119 427802.451)
988,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",26,56,2277,Arlington Special Educ School,"315 N French Ave, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),POINT (1326133.353 438805.808)
989,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",802,1784,3124,Post Middle School,"1220 E 5th St, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.19648800,-122.11136799,48.19650000,-122.11140000,10,0,POINT (1328729.118048057 439035.41536071897),POINT (1328729.118 439035.415)
990,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",805,1796,4154,Presidents Elementary School,"505 E 3rd St, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.19549399,-122.12042598,48.19550000,-122.12030000,10,0,POINT (1326552.7136082202 438706.85007297993),POINT (1326552.714 438706.85)
991,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",522,1099,4436,Kent Prairie Elementary School,"8110 207th St NE, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.18350799,-122.11942703,48.18360000,-122.11950000,10,0,POINT (1326675.254045561 434363.21505947411),POINT (1326675.254 434363.215)
992,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",24,54,2523,Arlington High School,"18821 Crown Ridge Blvd, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.16557701,-122.11526197,48.16620000,-122.11500000,10,0,POINT (1327667.1684003025 427998.4843506366),POINT (1327667.168 427998.484)
993,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",25,55,5495,Arlington Open Doors,"4407 172nd St NE, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.15614301,-122.16723204,48.15310600,-122.17017200,10,0,POINT (1314122.4182588905 423451.74081946909),POINT (1314122.418 423451.741)
994,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",1084,2462,4287,Weston High School,"4407 172nd St NE, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.15614301,-122.16723204,48.15310600,-122.17017200,10,0,POINT (1314122.4182588905 423451.74081946909),POINT (1314122.418 423451.741)
995,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",403,853,5010,Haller Middle School,"600 E 1st St, Arlington, WA 98223",...,Legislative District 10,Snohomish,48.19228800,-122.12119997,48.19160000,-122.12100000,10,0,POINT (1326358.2090759724 437287.22988072038),POINT (1326358.209 437287.23)
996,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",967,2167,1714,Stillaguamish Valley Learning Center,"121

In [236]:
def find_nearest(gdA, gdB):
    """ Find nearest value between two geodataframes.
        Returns "dist" for distance between nearest points.
    """
    nA = np.array(list(gdA.geometry.apply(lambda x: (x.x, x.y))))
    nB = np.array(list(gdB.geometry.apply(lambda x: (x.x, x.y))))
    
    btree = cKDTree(nB)
    dist, idx = btree.query(nA, k=1)
    gdB_nearest = gdB.iloc[idx].drop(columns="geometry").reset_index(drop=True)
    gdf = pd.concat(
        [
            gdA.reset_index(drop=True),
            gdB_nearest,
            pd.Series(dist, name='dist')
        ], 
        axis=1)

    return gdf

In [237]:
parcel_dict["Arlington School District"]

,Census2020Tract,parcelid,geometry,population,index_right,lea_name
843554,5.306105e+10,1062044,POINT (1325575.152 438403.362),2,43.0,Arlington School District
843555,5.306105e+10,1062045,POINT (1325573.807 438308.313),1,43.0,Arlington School District
843556,5.306105e+10,1062046,POINT (1325571.975 438251.368),5,43.0,Arlington School District
843557,5.306105e+10,1062047,POINT (1325570.329 438194.491),2,43.0,Arlington School District
843558,5.306105e+10,1062048,POINT (1325568.057 438118.53),4,43.0,Arlington School District
...,...,...,...,...,...,...
1053512,5.306105e+10,1325907,POINT (1368364.865 449437.94),2,43.0,Arlington School District
1053513,5.306105e+10,1325909,POINT (1369359.978 451248.461),4,43.0,Arlington School District
1053514,5.306105e+10,1325914,POINT (1367364.243 448382.979),2,43.0,Arlington School District
1053515,5.306105e+10,1325915,POINT (1367679.443 448379.128),2,43.0,Arlington School District


In [153]:
# print(school_dict["Arlington School District"].crs)
# print(parcel_dict["Arlington School District"].crs)

+init=epsg:2285 +type=crs


In [239]:
# parcel_dict = parcel_dict.copy()
# school_dict = school_dict.copy()
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].rename(columns={'geometry_x': 'geometry'})
school_dict["Arlington School District"] = school_dict["Arlington School District"].rename(columns={'school_geometry': 'geometry'})
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].set_geometry("geometry")
school_dict["Arlington School District"] = school_dict["Arlington School District"].set_geometry("geometry")

# school_dict["Arlington School District"] = school_dict["Arlington School District"].set_crs(epsg=4326, allow_override=True)
# parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].set_crs(epsg=4326, allow_override=True)

# school_dict["Arlington School District"] = school_dict["Arlington School District"].to_crs(epsg=2285)
# parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].to_crs(epsg=2285)

gdf = find_nearest(parcel_dict["Arlington School District"], school_dict["Arlington School District"])

In [240]:
gdf[["Census2020Tract", "parcelid", "population", "lea_name", "school", "dist"]]
gdf["miles"] = gdf['dist'] / 5280.0

In [242]:
gdf_small = gdf[["Census2020Tract", "parcelid", "population", "lea_name", "school", "dist", "miles"]]

In [243]:
gdf_small['wt_tot'] = gdf_small["miles"] * gdf_small["population"]

# Aggregate sums for each group
df_agg = gdf_small.groupby("Census2020Tract")[["population", 'wt_tot']].sum()

# Compute weighted average
df_agg['wt_avg'] = df_agg['wt_tot'] / df_agg["population"]

In [244]:
df_agg

,population,wt_tot,wt_avg
Census2020Tract,,,
5.306105e+10,1684,3931.508609,2.334625
5.306105e+10,697,3065.959678,4.398794
5.306105e+10,436,296.265654,0.679508
5.306105e+10,465,2004.575431,4.310915
5.306105e+10,577,3431.075838,5.946405
5.306105e+10,487,2599.286763,5.337344
5.306105e+10,5708,22147.481127,3.880077
5.306105e+10,1798,4499.386731,2.502440
5.306105e+10,4854,20869.514565,4.299447


In [245]:
results = []
for district_name in parcel_dict.keys():
    print(district_name)
    parcels = parcel_dict[district_name].copy()
    schools = school_dict[district_name].copy()
    
    parcels = parcels.rename(columns={'geometry_x': 'geometry'})
    schools = schools.rename(columns={'school_geometry': 'geometry'})

    # Make sure geometry column is active
    parcels = parcels.set_geometry("geometry")
    schools = schools.set_geometry("geometry")
    
    # Ensure CRS matches
    # schools = schools.to_crs(parcels.crs)
    
    # Compute nearest
    nearest_df = find_nearest(parcels, schools)
    
    # Collect
    results.append(nearest_df)

Arlington School District
Auburn School District
Bainbridge Island School District
Bellevue School District
Bethel School District
Bremerton School District
Carbonado School District
Central Kitsap School District
Clover Park School District
Darrington School District
Dieringer School District
Eatonville School District
Edmonds School District
Enumclaw School District
Everett School District
Federal Way School District
Fife School District
Franklin Pierce School District
Granite Falls School District
Highline School District
Index School District
Issaquah School District
Kent School District
Lake Stevens School District
Lake Washington School District
Lakewood School District
Marysville School District
Mercer Island School District
Monroe School District
Mukilteo School District
North Kitsap School District
Northshore School District
Orting School District
Peninsula School District
Puyallup School District
Renton School District
Riverview School District
Seattle Public Schools
Shorelin

In [246]:
results

[      Census2020Tract  parcelid                        geometry  population  \
 0        5.306105e+10   1062044  POINT (1325575.152 438403.362)           2   
 1        5.306105e+10   1062045  POINT (1325573.807 438308.313)           1   
 2        5.306105e+10   1062046  POINT (1325571.975 438251.368)           5   
 3        5.306105e+10   1062047  POINT (1325570.329 438194.491)           2   
 4        5.306105e+10   1062048   POINT (1325568.057 438118.53)           4   
 ...               ...       ...                             ...         ...   
 9908     5.306105e+10   1325907   POINT (1368364.865 449437.94)           2   
 9909     5.306105e+10   1325909  POINT (1369359.978 451248.461)           4   
 9910     5.306105e+10   1325914  POINT (1367364.243 448382.979)           2   
 9911     5.306105e+10   1325915  POINT (1367679.443 448379.128)           2   
 9912     5.306105e+10   1325921  POINT (1366379.408 448437.633)           3   
 
       index_right                   l

In [247]:
all_nearest_df = pd.concat(results, ignore_index=True)
all_nearest_df

,Census2020Tract,parcelid,geometry,population,index_right,lea_name,lea_code,short_name,lea_name_x,Shape_x,...,legislative,county,geocoded_y,geocoded_x,nces_y,nces_x,adjusted_l,SDE_STATE_ID,Shape_y,dist
0,5.306105e+10,1062044,POINT (1325575.152 438403.362),2,43.0,Arlington School District,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),688.149999
1,5.306105e+10,1062045,POINT (1325573.807 438308.313),1,43.0,Arlington School District,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),748.727206
2,5.306105e+10,1062046,POINT (1325571.975 438251.368),5,43.0,Arlington School District,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),789.017219
3,5.306105e+10,1062047,POINT (1325570.329 438194.491),2,43.0,Arlington School District,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),831.085679
4,5.306105e+10,1062048,POINT (1325568.057 438118.53),4,43.0,Arlington School District,31016,Arlington,Arlington School District,POLYGON ((1309152.9065443873 476285.6577892303...,...,Legislative District 10,Snohomish,48.19523484,-122.12253438,48.19575200,-122.12202600,10,0,POINT (1326133.3525703102 438805.80820839107),889.893638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1052307,5.305307e+10,1053553,POINT (1316396.678 56390.587),3,37.0,White River School District,27416,White River,White River School District,POLYGON ((1317390.6853443831 80729.97350847721...,...,Legislative District 31,Pierce,47.15990201,-122.11568300,47.16015800,-122.11573400,31,0,POINT (1321373.1307915598 61078.108612805605),6836.515029
1052308,5.305307e+10,1053554,POINT (1316385.696 56480.434),1,37.0,White River School District,27416,White River,White River School District,POLYGON ((1317390.6853443831 80729.97350847721...,...,Legislative District 31,Pierce,47.15990201,-122.11568300,47.16015800,-122.11573400,31,0,POINT (1321373.1307915598 61078.108612805605),6783.297478
1052309,5.305307e+10,1053555,POINT (1316423.672 56580.798),2,37.0,White River School District,27416,White River,White River School District,POLYGON ((1317390.6853443831 80729.97350847721...,...,Legislative District 31,Pierce,47.15990201,-122.11568300,47.16015800,-122.11573400,31,0,POINT (1321373.1307915598 61078.108612805605),6687.521034
1052310,5.305307e+10,1053556,POINT (1316441.219 56729.259),2,37.0,White River School District,27416,White River,White River School District,POLYGON ((1317390.6853443831 80729.97350847721...,...,Legislative District 31,Pierce,47.15990201,-122.11568300,47.16015800,-122.11573400,31,0,POINT (1321373.1307915598 61078.108612805605),6575.427462


In [248]:
all_nearest_df_small = all_nearest_df[["Census2020Tract", "parcelid", "population", "lea_name", "school", "dist"]]

In [249]:
all_nearest_df_small

,Census2020Tract,parcelid,population,lea_name,school,dist
0,5.306105e+10,1062044,2,Arlington School District,Arlington Special Educ School,688.149999
1,5.306105e+10,1062045,1,Arlington School District,Arlington Special Educ School,748.727206
2,5.306105e+10,1062046,5,Arlington School District,Arlington Special Educ School,789.017219
3,5.306105e+10,1062047,2,Arlington School District,Arlington Special Educ School,831.085679
4,5.306105e+10,1062048,4,Arlington School District,Arlington Special Educ School,889.893638
...,...,...,...,...,...,...
1052307,5.305307e+10,1053553,3,White River School District,Foothills Elementary School,6836.515029
1052308,5.305307e+10,1053554,1,White River School District,Foothills Elementary School,6783.297478
1052309,5.305307e+10,1053555,2,White River School District,Foothills Elementary School,6687.521034
1052310,5.305307e+10,1053556,2,White River School District,Foothills Elementary School,6575.427462


In [250]:
def weighted_avg(df, val_col, wt_col, agg_col):
    """ Returns weighted average for specified aggregation. 
        
        Parameters
    ----------
    df : Pandas DataFrame 
    val_col: column name of the value being averaged
    wt_col: weight column name
    agg_col: column to be used for aggregation
    ----------
    """
    df = df.copy()
    df['wt_tot'] = df[val_col] * df[wt_col]
    
    # Aggregate sums for each group
    df_agg = df.groupby(agg_col)[[wt_col, 'wt_tot']].sum()
    
    # Compute weighted average
    df_agg['wt_avg'] = df_agg['wt_tot'] / df_agg[wt_col]
    
    return df_agg

In [251]:
all_nearest_df_small['miles'] = all_nearest_df_small['dist'] / 5280.0
tract_output_df = weighted_avg(all_nearest_df_small, val_col='miles', wt_col='population', agg_col='Census2020Tract').reset_index()

In [252]:
tract_output_df

,Census2020Tract,population,wt_tot,wt_avg
0,5.303300e+10,3670,1678.395684,0.457329
1,5.303300e+10,4270,1555.169595,0.364208
2,5.303300e+10,4401,2068.027695,0.469899
3,5.303300e+10,3989,1707.980316,0.428173
4,5.303300e+10,2831,1296.394933,0.457928
...,...,...,...,...
914,5.306105e+10,3780,10743.377929,2.842163
915,5.306105e+10,7707,4532.616155,0.588117
916,5.306105e+10,5807,6358.518676,1.094975
917,5.306194e+10,6595,12940.328958,1.962142


In [253]:
tract_output_df.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\pop_avg_dist_to_schools.csv')